In [39]:
import pandas as pd 


df = pd.read_json("data/source/flore200.jsonl", lines=True)

In [40]:
lux_df = df[df["iso_639_3"] == "ltz"]
eng_df = df[df["iso_639_3"] == "eng"]

lux_df = df[df["iso_639_3"] == "ltz"][["id", "text","topic"]].copy()
eng_df = df[df["iso_639_3"] == "eng"][["id", "text","topic"]].copy()


lux_df["id"] = lux_df["id"].astype(int)
eng_df["id"] = eng_df["id"].astype(int)

df_joined = pd.merge(
    lux_df, eng_df,
    on=["id", "topic"],
    suffixes=("_ltz", "_eng"),
    how="inner"
)

df_joined.drop_duplicates(subset=["id"], inplace=True)
df_joined["source"]= "flore200"
df_joined = df_joined[["id", "text_ltz", "text_eng", "source"]]
df_joined.to_json("data/performance_eval/flore200_eng_lux_raw.jsonl", lines=True, orient="records")


In [44]:
val_300_df = pd.read_json("data/source/dataset_val_300.jsonl", lines=True)

In [45]:
val_300_df_new = pd.DataFrame()
val_300_df_new['id'] = val_300_df.index_unique
val_300_df_new['id'] = val_300_df_new['id'].astype(int)
val_300_df_new["source"] = "val_300"
val_300_df_new["text_ltz"] = val_300_df["input"]
val_300_df_new["text_eng"] = val_300_df["translated_text"]

In [48]:
val_300_df_new.to_json("data/performance_eval/val_300_eng_lux_raw.jsonl", lines=True, orient="records")

In [49]:
df_samples = pd.read_json("data/extraction_pdf/datasets/df_samples.jsonl", lines=True)

In [51]:
df_samples_new = pd.DataFrame()
df_samples_new['id'] = df_samples.idx_samples
df_samples_new["source"] = "samples_pdf"
df_samples_new["text_ltz"] = df_samples["luxembourg"]
df_samples_new["text_eng"] = df_samples["english"]

In [54]:
df_samples_new.to_json("data/performance_eval/samples_grammar_eng_lux_raw.jsonl", lines=True, orient="records")

In [55]:
df_flore200 = pd.read_json("data/performance_eval/flore200_eng_lux_raw.jsonl", lines=True)
df_val300 = pd.read_json("data/performance_eval/val_300_eng_lux_raw.jsonl", lines=True)
df_samples = pd.read_json("data/performance_eval/samples_grammar_eng_lux_raw.jsonl", lines=True)

In [64]:
df_concat = pd.concat([df_flore200, df_val300, df_samples], ignore_index=True).reset_index()


In [65]:
df_concat.drop(columns=["id"], inplace=True)
df_concat.rename(columns={"index": "id"}, inplace=True)

In [70]:
df_concat.text_eng.value_counts()

text_eng
Researchers have reported neighborhoods in which community centers, schools, and shops use different dominant languages, creating varied opportunities for everyday multilingual encounters among the residents.                                                            1
On Monday, scientists from the Stanford University School of Medicine announced the invention of a new diagnostic tool that can sort cells by type: a tiny printable chip that can be manufactured using standard inkjet printers for possibly about one U.S. cent each.    1
Lead researchers say this may bring early detection of cancer, tuberculosis, HIV and malaria to patients in low-income countries, where the survival rates for illnesses such as breast cancer can be half those of richer countries.                                       1
The JAS 39C Gripen crashed onto a runway at around 9:30 am local time (0230 UTC) and exploded, closing the airport to commercial flights.                                            

In [71]:
df_concat.to_json("data/performance_eval/eng_lux_concat.jsonl", lines=True, orient="records")

## Start Inference Testing

In [ ]:
import pandas as pd 
import os
import pickle
from datetime import datetime, timezone
from openai import OpenAI  # pip install openai
import nltk
from nltk.tokenize import sent_tokenize
import os
from pprint import pprint
import pandas as pd
import sacrebleu 
from mt_reasoning.utils import prompts_util, clients_util, eval_util
from tqdm import tqdm
import importlib
from dotenv import load_dotenv
import random
import string
import argparse
from comet import download_model, load_from_checkpoint
from typing import List, Dict
from huggingface_hub import snapshot_download, login

load_dotenv()



parser = argparse.ArgumentParser()
parser.add_argument('--model_vllm', type=str, default=os.environ.get("MODEL_VLLM", "/home/snt/projects_lujun/base_models/gemma-2-2b-it"), help='Path to the VLLM model')
parser.add_argument('--port', type=str, default=os.environ.get("VLLM_PORT", "1997"), help='Port for VLLM service')

args = parser.parse_args()

model_vllm = args.model_vllm
PORT = args.port

# model_vllm = "/home/snt/projects_lujun/base_models/gemma-2-2b-it"
# PORT = 1997

source_df = pd.read_json("data/performance_eval/eng_lux_concat.jsonl", lines=True)

## uv run vllm serve /home/snt/projects_lujun/base_models/gemma-2-2b-it --host 0.0.0.0 --port 1997 --max-model-len 2048 --max-num-seqs 2 --gpu-memory-utilization 0.7


importlib.reload(prompts_util)
importlib.reload(clients_util)

nltk.download('punkt')

## Open AI Settings
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")
TEMPERATURE = float(os.environ.get("OPENAI_TEMPERATURE", "0.5"))
HUGGINGFACE_TOKEN= os.environ.get("HUGGINGFACE_TOKEN", "")
login(token=HUGGINGFACE_TOKEN)
## VllM settings
# model_vllm = os.environ.get("MODEL_VLLM", "/home/snt/projects_lujun/base_models/gemma-2-2b-it")
IP = os.environ.get("VLLM_IP", "0.0.0.0")
# PORT = os.environ.get("VLLM_PORT", "1997")
server_url = f"http://{IP}:{PORT}/v1"
print (server_url)

if "gpt" in model_vllm:
    vllm_client = OpenAI(api_key=OPENAI_API_KEY)
    vllm_extra={"logprobs": False}
else:
    vllm_client = OpenAI(base_url=server_url)
    vllm_extra={"logprobs": False}

## Experimental Settings
# grammar_list_size = 5
letters = list(string.ascii_uppercase)  # ['A', 'B', 'C', ..., 'Z']

# vllm_extra={"logprobs": True, "top_logprobs": grammar_list_size}
time_now = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

project_dir = os.environ.get("PROJECT_DIR", None)
output_dir = os.path.join(project_dir, "data/performance_eval")

import glob
pattern = os.path.join(
    output_dir,
    f"task_eval_translation_*_{model_vllm.split('/')[-1]}.jsonl"
)

matches = glob.glob(pattern)
if matches:
    matches.sort(key=os.path.getmtime, reverse=True)
    output_path = matches[0]
    print(f"Resuming from existing file: {output_path}")
    finished_lines = len(pd.read_json(output_path, lines=True))
    print(f"Already processed {finished_lines} lines.")
else:
    finished_lines = 0
    output_path = os.path.join(output_dir, f"task_eval_translation_{time_now}_{model_vllm.split('/')[-1]}.jsonl")


comet_model = eval_util.init_comet_model(model_name="Unbabel/wmt23-cometkiwi-da-xl", gpus=1)
comet_model_small = eval_util.init_comet_model(model_name="Unbabel/wmt22-cometkiwi-da", gpus=1)
chrf_metric = sacrebleu.CHRF(word_order=3)

for index, row in tqdm(source_df.iterrows(), total=len(source_df)):
    if index < finished_lines:
        continue  # Skip already processed rows

    lux_sentence = row['text_ltz']
    eng_sentence = row['text_eng']

    input_dict = {
        "ENGLISH_SENTENCE": eng_sentence,
    }

    output_dict, input_prompt, log_probs = clients_util.generate_with_calling_api(
        client=vllm_client,
        system_prompt_template_path="prompts/system/system_prompt_translation.jinja",
        input_prompt_template_path="prompts/evaluation/prompt_translation_eng_lux.jinja",  # Use simple, complecated one confuse the models
        input_text_dict=input_dict,
        model=model_vllm,
        vllm_extra=vllm_extra
    )
    
    row["input_prompt"] = input_prompt
    row["task_translation_dict"] = output_dict
    translated_text = output_dict.get("translation", "").strip()

    # Cometkiwi
    comet_score = eval_util.evaluate_with_comet_ref(model=comet_model, src=[eng_sentence], mt=[translated_text], ref=[lux_sentence])["system_score"]
    comet_score_small = eval_util.evaluate_with_comet_ref(model=comet_model_small, src=[eng_sentence], mt=[translated_text], ref=[lux_sentence])["system_score"]

    # spbleu score
    spbleu_score = sacrebleu.corpus_bleu([translated_text], [[lux_sentence]], tokenize="flores200").score

    # Chrf++ score
    charf_score = chrf_metric.sentence_score(translated_text, [lux_sentence]).score

    row["CometScore"] = comet_score
    row["CometScoreSmall"] = comet_score_small
    row["spbleu_score"] = spbleu_score
    row["chrf_score"] = charf_score

    updated_row = pd.DataFrame([row])
    if not os.path.exists(output_dir):  
        os.makedirs(output_dir)
    if not os.path.exists(output_path):
        updated_row.to_json(output_path, orient="records", lines=True)
    else:
        updated_row.to_json(output_path, orient="records", lines=True, mode="a")
    
    print(output_dict)
    print("----------------------------------------------")
    pprint(output_dict, indent=2, width=150, sort_dicts=False)

[nltk_data] Downloading package punkt to /home/snt/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


http://0.0.0.0:1997/v1
Resuming from existing file: /home/snt/projects_lujun/mt_reasoning/data/performance_eval/task_eval_translation_20251005_225929_gemma-2-2b-it.jsonl
Already processed 4 lines.


Fetching 6 files: 100%|██████████| 6/6 [01:48<00:00, 18.11s/it]
Encoder model frozen.
  0%|          | 0/3352 [00:00<?, ?it/s]HTTP Request: POST http://0.0.0.0:1997/v1/chat/completions "HTTP/1.1 200 OK"
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/home/snt/projects_lujun/mt_reasoning/.venv/lib/python3.10/site-packages/pytorch_lightning/trainer/setup.py:177: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
  0%|          | 5/3352 [00:04<49:46,  1.12it/s]

{'translation': "L'Media lokale entrop floor acide a' L-Mobillet an 'a Finanzierour rouat vogue um side z'éis' et a'ne gorné 'üe"}
----------------------------------------------
{'translation': "L'Media lokale entrop floor acide a' L-Mobillet an 'a Finanzierour rouat vogue um side z'éis' et a'ne gorné 'üe"}


HTTP Request: POST http://0.0.0.0:1997/v1/chat/completions "HTTP/1.1 200 OK"
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/home/snt/projects_lujun/mt_reasoning/.venv/lib/python3.10/site-packages/pytorch_lightning/trainer/setup.py:177: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
  0%|          | 6/3352 [00:06<1:08:59,  1.24s/it]

...

{'translation': 'Zes minut nach 11:00, protesten haarden déi Traffike an de Nordeche in Whitehall blockiere'}
----------------------------------------------
{'translation': 'Zes minut nach 11:00, protesten haarden déi Traffike an de Nordeche in Whitehall blockiere'}


HTTP Request: POST http://0.0.0.0:1997/v1/chat/completions "HTTP/1.1 200 OK"
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/home/snt/projects_lujun/mt_reasoning/.venv/lib/python3.10/site-packages/pytorch_lightning/trainer/setup.py:177: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
  0%|          | 10/3352 [00:16<1:57:15,  2.11s/it]

{'translation': 'An De 11:20 h, de Police heescht de Protestierten, em zu entan weg vum Schτές, Ärgegsordde, wie a behebe Muss sin.  '}
----------------------------------------------
{'translation': 'An De 11:20 h, de Police heescht de Protestierten, em zu entan weg vum Schτές, Ärgegsordde, wie a behebe Muss sin.  '}


HTTP Request: POST http://0.0.0.0:1997/v1/chat/completions "HTTP/1.1 200 OK"
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/home/snt/projects_lujun/mt_reasoning/.venv/lib/python3.10/site-packages/pytorch_lightning/trainer/setup.py:177: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
  0%|          | 10/3352 [00:20<1:52:37,  2.02s/it]


KeyboardInterrupt: 

In [ ]:
import pandas as pd 
import os
import pickle
from datetime import datetime, timezone
from openai import OpenAI  # pip install openai
import nltk
from nltk.tokenize import sent_tokenize
import os
from pprint import pprint
import pandas as pd
import sacrebleu 
from mt_reasoning.utils import prompts_util, clients_util, eval_util
from tqdm import tqdm
import importlib
from dotenv import load_dotenv
import random
import string
import argparse
from comet import download_model, load_from_checkpoint
from typing import List, Dict
from huggingface_hub import snapshot_download, login

load_dotenv()



# parser = argparse.ArgumentParser()
# parser.add_argument('--model_vllm', type=str, default=os.environ.get("MODEL_VLLM", "/home/snt/projects_lujun/base_models/gemma-2-2b-it"), help='Path to the VLLM model')
# parser.add_argument('--port', type=str, default=os.environ.get("VLLM_PORT", "1997"), help='Port for VLLM service')

# args = parser.parse_args()

# model_vllm = args.model_vllm
# PORT = args.port

model_vllm = "/home/snt/projects_lujun/base_models/gemma-2-2b-it"
PORT = 1997

source_df = pd.read_json("data/performance_eval/eng_lux_concat.jsonl", lines=True)

## uv run vllm serve /home/snt/projects_lujun/base_models/gemma-2-2b-it --host 0.0.0.0 --port 1997 --max-model-len 2048 --max-num-seqs 2 --gpu-memory-utilization 0.7


importlib.reload(prompts_util)
importlib.reload(clients_util)

nltk.download('punkt')

## Open AI Settings
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")
TEMPERATURE = float(os.environ.get("OPENAI_TEMPERATURE", "0.5"))
HUGGINGFACE_TOKEN= os.environ.get("HUGGINGFACE_TOKEN", "")
login(token=HUGGINGFACE_TOKEN)
## VllM settings
# model_vllm = os.environ.get("MODEL_VLLM", "/home/snt/projects_lujun/base_models/gemma-2-2b-it")
IP = os.environ.get("VLLM_IP", "0.0.0.0")
# PORT = os.environ.get("VLLM_PORT", "1997")
server_url = f"http://{IP}:{PORT}/v1"
print (server_url)

if "gpt" in model_vllm:
    vllm_client = OpenAI(api_key=OPENAI_API_KEY)
    vllm_extra={"logprobs": False}
else:
    vllm_client = OpenAI(base_url=server_url)
    vllm_extra={"logprobs": False}

## Experimental Settings
# grammar_list_size = 5
letters = list(string.ascii_uppercase)  # ['A', 'B', 'C', ..., 'Z']

# vllm_extra={"logprobs": True, "top_logprobs": grammar_list_size}
time_now = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

project_dir = os.environ.get("PROJECT_DIR", None)
output_dir = os.path.join(project_dir, "data/performance_eval")

import glob
pattern = os.path.join(
    output_dir,
    f"task_eval_translation_*_{model_vllm.split('/')[-1]}.jsonl"
)

matches = glob.glob(pattern)
if matches:
    matches.sort(key=os.path.getmtime, reverse=True)
    output_path = matches[0]
    print(f"Resuming from existing file: {output_path}")
    finished_lines = len(pd.read_json(output_path, lines=True))
    print(f"Already processed {finished_lines} lines.")
else:
    finished_lines = 0
    output_path = os.path.join(output_dir, f"task_eval_translation_{time_now}_{model_vllm.split('/')[-1]}.jsonl")


print(f"Starting evaluation for model: {model_vllm}")
comet_model = eval_util.init_comet_model(model_name="Unbabel/wmt23-cometkiwi-da-xl", gpus=1)
comet_model_small = eval_util.init_comet_model(model_name="Unbabel/wmt22-cometkiwi-da", gpus=1)
chrf_metric = sacrebleu.CHRF(word_order=3)

for index, row in tqdm(source_df.iterrows(), total=len(source_df)):
    if index < finished_lines:
        continue  # Skip already processed rows

    lux_sentence = row['text_ltz']
    eng_sentence = row['text_eng']

    input_dict = {
        "ENGLISH_SENTENCE": eng_sentence,
    }

    output_dict, input_prompt, log_probs = clients_util.generate_with_calling_api(
        client=vllm_client,
        system_prompt_template_path="prompts/system/system_prompt_translation.jinja",
        input_prompt_template_path="prompts/evaluation/prompt_translation_eng_lux.jinja",  # Use simple, complecated one confuse the models
        input_text_dict=input_dict,
        model=model_vllm,
        vllm_extra=vllm_extra
    )
    
    row["input_prompt"] = input_prompt
    row["task_translation_dict"] = output_dict
    translated_text = output_dict.get("translation", "").strip()

    # Cometkiwi
    comet_score = eval_util.evaluate_with_comet_ref(model=comet_model, src=[eng_sentence], mt=[translated_text], ref=[lux_sentence])["system_score"]
    comet_score_small = eval_util.evaluate_with_comet_ref(model=comet_model_small, src=[eng_sentence], mt=[translated_text], ref=[lux_sentence])["system_score"]

    # spbleu score
    spbleu_score = sacrebleu.corpus_bleu([translated_text], [[lux_sentence]], tokenize="flores200").score

    # Chrf++ score
    charf_score = chrf_metric.sentence_score(translated_text, [lux_sentence]).score

    row["CometScore"] = comet_score
    row["CometScoreSmall"] = comet_score_small
    row["spbleu_score"] = spbleu_score
    row["chrf_score"] = charf_score

    updated_row = pd.DataFrame([row])
    if not os.path.exists(output_dir):  
        os.makedirs(output_dir)
    if not os.path.exists(output_path):
        updated_row.to_json(output_path, orient="records", lines=True)
    else:
        updated_row.to_json(output_path, orient="records", lines=True, mode="a")
    
    # print(output_dict)
    # print("----------------------------------------------")
    # pprint(output_dict, indent=2, width=150, sort_dicts=False)